# M32 lab solution: Feedback flywheel

**Brief.** Simulate 100 thumbs-up / thumbs-down events and show the effect on ranking for the affected queries.

The events are simulated with a fixed seed over the 25 golden questions that have evidence. Each event is a user rating one of the top 4 passages shown for a question: up when the passage holds an evidence span, down otherwise, with one rating in ten flipped as noise. The reranking follows lesson 32.2: feedback scales the retrieval score of the passage it rated. No model is called; only the embeddings are.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

## 1. The baseline ranking and the target passage of each question

In [ ]:
import random

K_CANDIDATES, K_SHOWN = 8, 4
answerable = [g for g in golden if g["evidence_spans"]]
chunk_id = {d.page_content: i for i, d in enumerate(chunks)}
candidates = {g["id"]: [(chunk_id[d.page_content], s) for d, s in store.similarity_search_with_score(g["question"], k=K_CANDIDATES)]
              for g in answerable}
# The target is the first chunk, in handbook order, that holds the question's first evidence span
target = {g["id"]: next(i for i, d in enumerate(chunks) if is_hit(d.page_content, g["evidence_spans"][:1])) for g in answerable}


def rank_of(ranked, tid):
    ids = [i for i, _ in ranked]
    return ids.index(tid) + 1 if tid in ids else None


print(pd.DataFrame([{"id": g["id"], "target rank (of 8)": rank_of(candidates[g["id"]], target[g["id"]])} for g in answerable])
      .to_string(index=False))

## 2. 100 simulated feedback events, seed 42

In [ ]:
SEED, N_EVENTS, NOISE = 42, 100, 0.1
rng = random.Random(SEED)
events = []
for _ in range(N_EVENTS):
    g = rng.choice(answerable)
    cid, _ = rng.choice(candidates[g["id"]][:K_SHOWN])
    helpful = is_hit(chunks[cid].page_content, g["evidence_spans"])
    if rng.random() < NOISE:
        helpful = not helpful
    events.append({"query": g["id"], "chunk": cid, "vote": 1 if helpful else -1})
events = pd.DataFrame(events)
print(events.vote.map({1: "up", -1: "down"}).value_counts().to_string())
print(events["query"].nunique(), "queries received feedback")

## 3. Feedback-weighted reranking

A passage's score is multiplied by `1 + 0.1 x net votes`, clipped to 0.5 to 1.5. Per-query feedback uses only the votes cast on that query; global feedback pools every vote on the passage, whichever query it came from.

In [ ]:
ALPHA, LOW, HIGH = 0.1, 0.5, 1.5
per_query = events.groupby(["query", "chunk"]).vote.sum().to_dict()
pooled = events.groupby("chunk").vote.sum().to_dict()


def rerank(qid, votes_for):
    scored = [(cid, s * min(HIGH, max(LOW, 1 + ALPHA * votes_for(qid, cid)))) for cid, s in candidates[qid]]
    return sorted(scored, key=lambda x: -x[1])


by_query = lambda qid, cid: per_query.get((qid, cid), 0)
by_chunk = lambda qid, cid: pooled.get(cid, 0)

## 4. Results: rank of the target passage before and after, per affected query

In [ ]:
rows = []
for g in answerable:
    qid = g["id"]
    rows.append({"id": qid, "events": int((events["query"] == qid).sum()),
                 "rank before": rank_of(candidates[qid], target[qid]),
                 "rank after (per-query)": rank_of(rerank(qid, by_query), target[qid]),
                 "rank after (global)": rank_of(rerank(qid, by_chunk), target[qid])})
ranks = pd.DataFrame(rows)
affected = ranks[ranks.events > 0]
print(affected.to_string(index=False))


def mrr(col, frame):
    return frame[col].apply(lambda r: 1 / r if r else 0).mean()


summary = pd.DataFrame([{"queries": label, "n": len(frame), **{c: round(mrr(c, frame), 3) for c in
                         ["rank before", "rank after (per-query)", "rank after (global)"]}}
                        for label, frame in [("affected", affected), ("not affected", ranks[ranks.events == 0])]])
print("\nMRR of the target passage (within the 8 candidates):")
print(summary.to_string(index=False))

## What this shows

- The per-query table is the brief's deliverable: for each query that received feedback, where its target passage ranked before and after.
- Per-query feedback moves only the queries that were rated. Pooled feedback also moves queries nobody rated, because a passage voted down for one question may be the answer to another; the "not affected" row shows whether that helped or hurt here.
- The noise rate and the seed are parameters. With one vote in ten flipped, a passage with a single vote can move the wrong way; set `NOISE = 0` to see the effect of clean feedback.